In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [ ]:
!pip install -U keras
!pip install -q tensorflow_datasets
!pip install -q -U tensorflow-text tensorflow
!pip install datasets pyarrow pillow matplotlib
!pip install onnx tf2onnx onnxruntime

In [ ]:
import os
import tensorflow as tf
from pathlib import Path

MODELNAME = "model.keras"

In [ ]:
from datasets import load_dataset

dataset = load_dataset("AMLP10101/human-segmentation-dataset")
dataset.save_to_disk("/content/dataset/")


In [ ]:
def load_image(path, size=[1280, 1918]):
    if not isinstance(path, tf.Tensor):
        path = str(path)

    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, size)
    return img


def procces_mask(path, size=[1280, 1918]):
    if not isinstance(path, tf.Tensor):
        path = str(path)

    raw = tf.io.read_file(path)
    mask = tf.io.decode_image(raw, channels=3, expand_animations=False)
    mask = tf.image.rgb_to_grayscale(mask)
    mask = tf.image.convert_image_dtype(mask, tf.float32)
    mask = tf.image.resize(mask, size, method="nearest")
    mask = tf.where(mask > 0.5, 1.0, 0.0)
    return mask

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from datasets import Image as HFImage
from PIL import Image

images = dataset.filter(lambda x: x["label"] == 1)
masks = dataset.filter(lambda x: x["label"] == 0)

print("Images:", len(images))
print("Masks:", len(masks))

image = np.array(images["train"][0]["image"].convert("RGB"))
mask = np.array(masks["train"][0]["image"].convert("L"))

mask = np.array(
    Image.fromarray(mask).resize(
        (image.shape[1], image.shape[0]),
        Image.Resampling.NEAREST
    )
)

overlay = image.copy()

overlay[mask > 127] = (
    0.6 * image[mask > 127] +
    0.4 * np.array([255, 0, 0])
).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

ax[0].imshow(image)
ax[0].set_title("Original image")

ax[1].imshow(mask, cmap="gray")
ax[1].set_title("Ground Truth")

ax[2].imshow(overlay)
ax[2].set_title("Overlay")

for a in ax:
    a.axis("off")

plt.show()

In [ ]:
from datasets import load_dataset
from pathlib import Path
from PIL import Image
import numpy as np
import tensorflow as tf
import os

dataset = load_dataset("AMLP10101/human-segmentation-dataset")["train"]

images = dataset.filter(lambda x: x["label"] == 1)
masks = dataset.filter(lambda x: x["label"] == 0)

OUTPUT_DIR = Path("overlay_check")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Изображений: {len(images)}")
print(f"Масок: {len(masks)}")

alpha = 0.25

for i in range(len(images)):

    image = np.array(images[i]["image"].convert("RGB"))
    mask = np.array(masks[i]["image"].convert("L"))

    mask = np.array(
        Image.fromarray(mask).resize(
            (image.shape[1], image.shape[0]),
            Image.Resampling.NEAREST
        )
    )

    image_tf = tf.convert_to_tensor(image, dtype=tf.float32) / 255.0

    mask_tf = tf.convert_to_tensor(mask > 127)

    red_layer = tf.ones_like(image_tf) * [1.0, 0.0, 0.0]

    blended = (1.0 - alpha) * image_tf + alpha * red_layer

    overlay_result = tf.where(
        mask_tf[:, :, None],
        blended,
        image_tf
    )

    overlay_uint8 = tf.image.convert_image_dtype(
        overlay_result,
        tf.uint8
    )

    encoded_png = tf.io.encode_png(overlay_uint8)

    save_path = OUTPUT_DIR / f"check_{i:04d}.png"

    tf.io.write_file(str(save_path), encoded_png)

print(f"Все наложения сохранены: {OUTPUT_DIR}")

In [ ]:
import math
import json
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

files = sorted(list(OUTPUT_DIR.glob("check_*.png")))
SAVE_FILE = Path("bad_masks_checkpoint.json")

IMAGES = 20
total_pages = math.ceil(len(files) / IMAGES)

if SAVE_FILE.exists():
    with open(SAVE_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
        PAGE = data.get("page", 0)
        bad_indexes = data.get("bad_indexes", [])
    print(f"Восстановлен прогресс: начинаем со страницы {PAGE + 1}/{total_pages}")
else:
    PAGE = 0
    bad_indexes = []

while True:
    start = PAGE * IMAGES
    batch = files[start : start + IMAGES]

    if not batch:
        print("\nВсе изображения проверены!")
        break

    cols = 5
    rows = math.ceil(len(batch) / cols)

    fig = plt.figure(figsize=(18, 12))
    for i, path in enumerate(batch):
        plt.subplot(rows, cols, i + 1)
        with Image.open(path) as img:
            plt.imshow(img)
        plt.title(path.stem.replace("check_", ""), fontsize=10)
        plt.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    user_input = input(
        f"[{PAGE + 1}/{total_pages}] 'q' - выход, 'b' - назад, Enter - всё ок: "
    ).strip()

    if user_input.lower() in ["q", "quit", "exit", "стоп"]:
        print("Прогресс сохранен.")
        break

    if user_input.lower() in ["b", "back", "назад"]:
        if PAGE > 0:
            PAGE -= 1
            print(f"Возврат на страницу {PAGE + 1}")
        else:
            print("Вы уже на первой странице")
        continue

    if user_input:
        bad_indexes.extend(user_input.split())

    PAGE += 1

    with open(SAVE_FILE, "w", encoding="utf-8") as f:
        json.dump({"page": PAGE, "bad_indexes": bad_indexes}, f, ensure_ascii=False, indent=2)

    print(f"Всего найдено плохих масок: {len(bad_indexes)}")

In [ ]:
from datasets import Image as HFImage
from pathlib import Path

raw_dataset = dataset.cast_column(
    "image",
    HFImage(decode=False)
)

image_files = {}
mask_files = {}

for i in range(len(raw_dataset)):
    item = raw_dataset[i]
    path = Path(item["image"]["path"])
    stem = path.stem

    if item["label"] == 1:
        image_files[stem] = i
    else:
        mask_files[stem] = i

print("Images:", len(image_files))
print("Masks:", len(mask_files))
print("Примеры:", list(image_files.keys())[:5])

In [ ]:
import json
import numpy as np
import tensorflow as tf
from PIL import Image

IMG_SIZE = 256
BATCH_SIZE = 16
VAL_SPLIT = 0.1
SEED = 42

# Формируем пары по именам файлов
# Исключаем вручную отмеченные плохие маски (ячейка с bad_masks_checkpoint.json)
bad_stems = set()
try:
    with open("bad_masks_checkpoint.json", encoding="utf-8") as f:
        bad_idx = set(int(x) for x in json.load(f)["bad_indexes"])

    k = 0
    for i in range(len(raw_dataset)):
        item = raw_dataset[i]
        if item["label"] == 1:
            if k in bad_idx:
                bad_stems.add(Path(item["image"]["path"]).stem)
            k += 1
    print(f"Исключено плохих масок: {len(bad_stems)}")
except FileNotFoundError:
    print("bad_masks_checkpoint.json не найден — используем все пары")

pairs = []

for name, image_idx in image_files.items():
    if name in mask_files and name not in bad_stems:
        pairs.append((image_idx, mask_files[name]))

print("Найдено пар:", len(pairs))

assert len(pairs) > 0, "Пары изображений не найдены!"
if not bad_stems:
    assert len(pairs) == len(image_files) == len(mask_files), \
        "Есть несопоставленные изображения или маски!"

# ---- Разделение на train / val (для честного контроля качества) ----
rng = np.random.default_rng(SEED)
order = rng.permutation(len(pairs))

val_count = max(1, int(len(pairs) * VAL_SPLIT))
val_pairs = [pairs[i] for i in order[:val_count]]
train_pairs = [pairs[i] for i in order[val_count:]]

print(f"Train: {len(train_pairs)}  Val: {len(val_pairs)}")



# ---- Кэш в памяти: один раз декодируем и ужимаем до IMG_SIZE ----
# Так обучение не тормозит на декодировании огромных исходников,
# а аугментации можно делать тяжёлыми.
def load_pair(image_idx, mask_idx):
    img = dataset[image_idx]["image"].convert("RGB").resize(
        (IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    msk = dataset[mask_idx]["image"].convert("L").resize(
        (IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    return (np.asarray(img, dtype=np.uint8),
            (np.asarray(msk) > 127).astype(np.float32)[..., None])


def build_arrays(pairs_list):
    xs, ys = zip(*(load_pair(i, m) for i, m in pairs_list))
    return np.stack(xs), np.stack(ys)


x_train, y_train = build_arrays(train_pairs)
x_val, y_val = build_arrays(val_pairs)
print("x_train:", x_train.shape, "y_train:", y_train.shape)


# ---- Аугментация (под веб-камеру) ----
# Геометрия должна быть ОДИНАКОВОЙ для изображения и маски, поэтому маска
# временно приклеивается к изображению 4-м каналом.
# rot90 убран: в веб-камере человек не лежит на боку.
# Ширина и высота зумятся независимо -> случайное изменение пропорций
# (кадр 16:9 в приложении сжимается в квадрат).
geometric_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.03, fill_mode="nearest"),
    tf.keras.layers.RandomZoom(
        height_factor=(-0.25, 0.15),
        width_factor=(-0.25, 0.25),
        fill_mode="nearest",
    ),
    tf.keras.layers.RandomTranslation(0.1, 0.1, fill_mode="nearest"),
])


def random_background():
    # Случайный гладкий цветной фон (низкочастотный шум, растянутый до кадра)
    small = tf.random.uniform([6, 6, 3], 0.0, 1.0)
    bg = tf.image.resize(small[None], [IMG_SIZE, IMG_SIZE], method="bicubic")[0]
    return tf.clip_by_value(bg * tf.random.uniform([1, 1, 3], 0.6, 1.0), 0.0, 1.0)


def augment(image, mask):
    # 1) Замена фона на синтетический (30%): учит не опираться на фон
    swap = tf.random.uniform([]) < 0.3
    image = tf.where(swap, image * mask + random_background() * (1.0 - mask), image)

    # 2) Геометрия (одинаково для изображения и маски)
    combined = tf.concat([image, mask], axis=-1)
    combined = geometric_aug(combined, training=True)
    image = combined[..., :3]
    mask = tf.where(combined[..., 3:] > 0.5, 1.0, 0.0)

    # 3) Цвет и свет (только изображение)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.7, 1.3)
    image = tf.image.random_saturation(image, 0.7, 1.3)
    image = tf.image.random_hue(image, 0.03)
    image = tf.clip_by_value(image, 0.0, 1.0)
    image = image * tf.random.uniform([1, 1, 3], 0.9, 1.1)          # баланс белого
    image = tf.pow(tf.clip_by_value(image, 1e-4, 1.0),
                   tf.random.uniform([], 0.7, 1.4))                   # гамма

    # 4) Артефакты камеры: размытие, шум, JPEG
    blurred = tf.nn.avg_pool2d(image[None], 3, 1, "SAME")[0]
    image = tf.where(tf.random.uniform([]) < 0.3, blurred, image)
    image = image + tf.random.normal(tf.shape(image),
                                     stddev=tf.random.uniform([], 0.0, 0.04))
    image = tf.clip_by_value(image, 0.0, 1.0)
    image = tf.cond(
        tf.random.uniform([]) < 0.5,
        lambda: tf.image.random_jpeg_quality(image, 40, 95),
        lambda: image,
    )

    image = tf.clip_by_value(image, 0.0, 1.0)
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return image, mask


def to_float(x, y):
    return tf.cast(x, tf.float32) / 255.0, y


train_data = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), reshuffle_each_iteration=True)
    .map(to_float, num_parallel_calls=tf.data.AUTOTUNE)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_data = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .map(to_float, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
# Проверка аугментации: изображение и маска должны оставаться согласованными
for batch_images, batch_masks in train_data.take(1):
    fig, axes = plt.subplots(4, 3, figsize=(12, 16))

    for i in range(4):
        img = batch_images[i].numpy()
        msk = batch_masks[i, :, :, 0].numpy()

        overlay = img.copy()
        overlay[msk > 0.5] = (
            0.6 * img[msk > 0.5] + 0.4 * np.array([1.0, 0.0, 0.0])
        )

        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Augmented image")
        axes[i, 1].imshow(msk, cmap="gray")
        axes[i, 1].set_title("Augmented mask")
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Overlay")

        for j in range(3):
            axes[i, j].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import tensorflow as tf

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

layer_names = [
    "block_1_expand_relu",
    "block_3_expand_relu",
    "block_6_expand_relu",
    "block_13_expand_relu",
    "block_16_project",
]

base_model_outputs = [
    base_model.get_layer(name).output
    for name in layer_names
]

down_stack = tf.keras.Model(
    inputs=base_model.input,
    outputs=base_model_outputs
)

down_stack.trainable = False


def upsample(filters, size):
    # kernel=4 + strides=2 даёт симметричный паддинг:
    # такой Conv2DTranspose поддерживается OpenCV DNN (cv::dnn::readNetFromONNX)
    return tf.keras.Sequential([
        tf.keras.layers.Conv2DTranspose(
            filters,
            size,
            strides=2,
            padding="same",
            use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.ReLU()
    ])

In [ ]:
def build_model():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # MobileNetV2 ожидает значения от -1 до 1
    x = inputs * 2.0 - 1.0

    skips = down_stack(x)

    x = skips[-1]
    skips = reversed(skips[:-1])

    filters = [512, 256, 128, 64]

    for up, skip, filters_count in zip(
        [upsample(f, 4) for f in filters],
        skips,
        filters
    ):
        x = up(x)
        x = tf.keras.layers.Concatenate()([x, skip])

    # Последнее увеличение 64x64 -> 128x128
    x = upsample(32, 4)(x)

    outputs = tf.keras.layers.Conv2D(
        1,
        kernel_size=1,
        activation="sigmoid"
    )(x)

    return tf.keras.Model(inputs, outputs)


model = build_model()

print("input :", model.input_shape)
print("output:", model.output_shape)
assert model.output_shape[1:3] == (IMG_SIZE, IMG_SIZE), \
    f"Выход модели {model.output_shape} не совпадает с размером маски {IMG_SIZE}x{IMG_SIZE}"

model.summary()

In [ ]:
def dice_loss(y_true, y_pred):
    smooth = 1e-6

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = (
        tf.reduce_sum(y_true, axis=[1, 2, 3])
        + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    )

    dice = (2.0 * intersection + smooth) / (union + smooth)

    return 1.0 - tf.reduce_mean(dice)


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)


def mean_iou(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = (
        tf.reduce_sum(y_true, axis=[1, 2, 3])
        + tf.reduce_sum(y_pred, axis=[1, 2, 3])
        - intersection
    )

    return tf.reduce_mean((intersection + 1e-6) / (union + 1e-6))


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=combined_loss,
    metrics=[mean_iou],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_mean_iou",
        mode="max",
        patience=12,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_mean_iou",
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_mean_iou",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=60,
    callbacks=callbacks,
)

model.save("model.keras")

In [ ]:
# ===== Fine-tuning энкодера =====
# Энкодер MobileNetV2 был заморожен — размораживаем его верхние блоки
# и дообучаем всю сеть на маленьком learning rate.
FINE_TUNE = True

if FINE_TUNE:
    down_stack.trainable = True

    fine_tune_from = "block_10_expand"
    trainable_flag = False

    for layer in down_stack.layers:
        if layer.name == fine_tune_from:
            trainable_flag = True
        # BatchNormalization лучше оставить замороженным
        layer.trainable = trainable_flag and not isinstance(
            layer, tf.keras.layers.BatchNormalization
        )

    print("Обучаемых параметров:", sum(
        int(tf.size(w)) for w in model.trainable_weights
    ))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
        loss=combined_loss,
        metrics=[mean_iou],
    )

    callbacks_fine = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_mean_iou",
            mode="max",
            patience=10,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_mean_iou",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            "best_model.keras",
            monitor="val_mean_iou",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
    ]

    history_fine = model.fit(
        train_data,
        validation_data=val_data,
        epochs=40,
        callbacks=callbacks_fine,
    )

    model.save("model.keras")

In [ ]:
# Кривые обучения
def plot_history(histories, labels):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    for h, label in zip(histories, labels):
        plt.plot(h.history["loss"], label=f"{label} train")
        plt.plot(h.history["val_loss"], "--", label=f"{label} val")
    plt.title("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    for h, label in zip(histories, labels):
        plt.plot(h.history["mean_iou"], label=f"{label} train")
        plt.plot(h.history["val_mean_iou"], "--", label=f"{label} val")
    plt.title("Mean IoU")
    plt.legend()

    plt.tight_layout()
    plt.show()


histories = [history]
labels = ["decoder"]

if "history_fine" in dir():
    histories.append(history_fine)
    labels.append("fine-tune")

plot_history(histories, labels)

In [ ]:
# Подбор оптимального порога бинаризации на валидационной выборке
best_t, best_iou = 0.5, 0.0

for t in np.arange(0.30, 0.70, 0.05):
    ious = []
    for imgs, msks in val_data:
        preds = model.predict(imgs, verbose=0)
        ious.append(mean_iou(msks, tf.cast(preds > t, tf.float32)).numpy())
    iou = float(np.mean(ious))
    print(f"threshold={t:.2f}  val IoU={iou:.4f}")
    if iou > best_iou:
        best_iou, best_t = iou, float(t)

print(f"\nЛучший порог: {best_t:.2f}, val IoU: {best_iou:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import os

# Если есть чекпоинт с лучшим val_loss — берём его
MODEL_PATH = "/content/best_model.keras"
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = f"/content/{MODELNAME}"

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("Модель успешно загружена:", MODEL_PATH)


def predict_image(image_path, alpha=0.35, threshold=0.5, use_tta=True):

    original = tf.keras.utils.load_img(image_path)

    original_np = np.array(original.convert("RGB"))

    resized = tf.image.resize(
        original_np,
        [IMG_SIZE, IMG_SIZE]
    )

    input_image = tf.cast(resized, tf.float32) / 255.0
    input_image = input_image[None, ...]

    prediction = model.predict(
        input_image,
        verbose=0
    )[0, :, :, 0]

    if use_tta:
        # Test-time augmentation: усредняем предсказание
        # с предсказанием для зеркально отражённого кадра
        pred_flip = model.predict(
            input_image[:, :, ::-1, :],   # горизонтальный флип (ось W)
            verbose=0
        )[0, :, :, 0]
        prediction = (prediction + pred_flip[:, ::-1]) / 2.0

    mask = (prediction > threshold).astype(np.uint8)

    mask = tf.image.resize(
        mask[..., None],
        [original_np.shape[0], original_np.shape[1]],
        method="nearest"
    ).numpy()[:, :, 0]

    mask = mask > 0.5

    overlay = original_np.copy()

    red = np.array([255, 0, 0])

    overlay[mask] = (
        (1 - alpha) * original_np[mask] +
        alpha * red
    ).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(original_np)
    axes[0].set_title("Original")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Predicted Mask")

    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")

    for ax in axes:
        ax.axis("off")

    plt.show()


In [ ]:
predict_image("/content/1.jpeg", alpha=0.24, threshold=best_t if "best_t" in dir() else 0.5)
# predict_image("/content/2.jpeg", alpha=0.4)


In [ ]:
import tensorflow as tf
import keras
import tf2onnx
import onnx

MODELNAME = "/content/model.keras"
OUTPUT_NAME = "/content/model.onnx"

# Загружаем без loss и optimizer
model = keras.models.load_model(
    MODELNAME,
    compile=False
)

print("Модель загружена!")

# Конвертируем
onnx_model, _ = tf2onnx.convert.from_keras(
    model,
    opset=13
)

# Сохраняем
onnx.save_model(
    onnx_model,
    OUTPUT_NAME
)

print(f"ONNX сохранён: {OUTPUT_NAME}")

In [ ]:
# Проверка: ONNX должен давать то же, что и Keras (иначе проблема в конвертации)
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession(OUTPUT_NAME, providers=["CPUExecutionProvider"])
inp, out = sess.get_inputs()[0], sess.get_outputs()[0]
print("ONNX input :", inp.name, inp.shape)    # ожидаем [?, 256, 256, 3]
print("ONNX output:", out.name, out.shape)    # ожидаем [?, 256, 256, 1]

xb, _ = next(iter(val_data))
x = xb[:4].numpy()
keras_pred = model.predict(x, verbose=0)
onnx_pred = sess.run(None, {inp.name: x})[0]
print("max abs diff Keras vs ONNX:", float(np.abs(keras_pred - onnx_pred).max()))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir /content/drive/MyDrive/HumanSegmentation/

!cp /content/model.keras /content/model.onnx /content/drive/MyDrive/HumanSegmentation/